# Axion PINN V2 — Physics-Faithful Implementation

## การปรับปรุงจาก `naxion.py` / `eoms.py`

### สมการที่ถูกต้องจาก `Code_CP`

**Klein-Gordon** (จาก `eoms.py: deriv_wfromphi`):
$$\ddot\phi_i = -\sqrt{3}\,\sqrt{\rho_{\rm tot}}\,\dot\phi_i - m_i^2\,\phi_i$$

**Friedmann** (scale factor evolution, จาก `eoms.py`):
$$\dot a = \frac{1}{\sqrt{3}}\,a\,\sqrt{\rho_{\rm ax} + \rho_m/a^3 + \rho_r/a^4 + \rho_\Lambda}$$

**ρ_total สำหรับ Hubble:**
$$H = \frac{\dot a}{a} = \frac{1}{\sqrt{3}}\,\sqrt{\rho_{\rm ax} + \rho_m/a^3 + \rho_r/a^4 + \rho_\Lambda}$$

**ρ_axion:**
$$\rho_i = \frac{1}{2}\dot\phi_i^2 + \frac{1}{2}m_i^2\phi_i^2$$

---

**Parameters จาก `configuration_card.ini`:**
- `rho_matter_0 = 0.81`, `rho_radiation_0 = 0.00027138`, `rho_lambda = 2.19`
- `a_in = 1e-8`, `t_in = 0`, `t_fi = 1`
- `phi_in_range = π`, `phidot_in = 0`

---

### การแก้ไขหลัก

| ปัญหา (V1 Advanced) | แก้ใน V2 |
|---|---|
| `ScaleFactorNet` init ผิด (ȧ(0)≈0 ไม่ใช่ H·a) | ใช้ **log-time input** + init bias ที่ถูกต้อง |
| Fourier features ใน linear-time ครอบ `[1e-8, 1]` ไม่ได้ | ใช้ **`τ = log(1 + t/t_scale)`** เป็น network input |
| WKB pretraining ไม่ถูกเรียก ใน `run_wkb()` | เรียก `train_advanced()` แทน `train_single()` |
| API bug ใน Exp D & E | สร้าง `run_advanced()` ที่ถูกต้อง |
| N=1 axion only | รองรับ N-axion เหมือน naxion.py |

## 1. Imports & Setup

In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float64
EPS    = 1e-20

print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

## 2. Cosmological Parameters (จาก configuration_card.ini)

In [ ]:
# ── Physical parameters from configuration_card.ini ──────────────────────────
RHO_M0   = 0.81          # rho_matter_0
RHO_R0   = 0.00027138    # rho_radiation_0
RHO_L0   = 2.19          # rho_lambda
A_IN     = 1e-8          # a_in
T_IN     = 0.0           # t_in
T_FI     = 1.0           # t_fi

# ── Useful constants ─────────────────────────────────────────────────────────
_INV_SQRT3 = 3.0**(-0.5)
_SQRT3     = 3.0**( 0.5)

# log-time scale: t_scale is chosen so τ = log(1 + t/t_scale) spans O(1)
# We use t_scale = a_in so τ ∈ [0, log(1 + T_FI/A_IN)] ≈ [0, 18.4]
T_SCALE = A_IN   # 1e-8

def to_log_time(t_tensor: torch.Tensor, t_scale: float = T_SCALE) -> torch.Tensor:
    """Map t ∈ [0, T] → τ = log(1 + t/t_scale), covering [0, ~18]."""
    return torch.log1p(t_tensor / t_scale)


def analyze_mass_regime(m_target: float, t_end: float = T_FI) -> dict:
    """Auto-select direct/rescaling strategy."""
    MAX_N_OSC = 500
    n_osc = m_target / (2 * np.pi) * t_end
    a_osc = (3 * RHO_R0 / m_target**2) ** 0.25

    if n_osc <= MAX_N_OSC:
        mode, m_sim, lam = 'direct', m_target, 1.0
        t_end_sim = t_end
    else:
        C     = 2 * np.pi * (MAX_N_OSC / 2) / t_end
        m_sim = max(100.0, (C * m_target**0.5) ** (2.0/3.0))
        lam   = np.sqrt(m_target / m_sim)
        mode  = 'rescaling'
        t_end_sim = t_end / lam

    regime = ('slow-roll'   if m_target < 3    else
               'onset'      if m_target < 50   else
               'oscillating' if m_target < 500 else 'rapid-osc')
    return dict(mode=mode, regime=regime, m_target=m_target, m_sim=m_sim,
                lam=lam, t_end=t_end, t_end_sim=t_end_sim,
                a_osc=a_osc, n_osc=n_osc)


def print_mass_summary(mc):
    print(f"{'─'*58}")
    print(f"  m_target={mc['m_target']:.3g}  regime={mc['regime']}  mode={mc['mode']}")
    print(f"  N_osc(physical)={mc['n_osc']:.2e}  |  a_osc={mc['a_osc']:.3e}")
    if mc['mode'] == 'rescaling':
        print(f"  m_sim={mc['m_sim']:.3g}  λ={mc['lam']:.3g}  t_end_sim={mc['t_end_sim']:.3e}")
    print(f"{'─'*58}")

## 3. ODE Reference Solver (faithful to `eoms.py`)

จาก `deriv_wfromphi` ใน `eoms.py`:
- `ȧ = (1/√3) √(ρ_ax·a² + ρ_m/a + ρ_r/a² + ρ_Λ·a²)` (equation สำหรับ scale factor)
- `φ̈ = -√3·√(ρ_tot)·φ̇ - m²·φ` โดย ρ_tot = ρ_ax + ρ_m/a³ + ρ_r/a⁴ + ρ_Λ

In [ ]:
def _one_axion_rhs(t, y, m2, rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0):
    """
    Single-axion equations of motion (faithful to eoms.py: deriv_wfromphi).

    State vector: y = [a, φ, φ̇]

    Equations:
      ȧ       = (1/√3) · a · √(rho_tot)
      dφ/dt   = φ̇
      dφ̇/dt   = -√3 · √(rho_tot) · φ̇  - m² · φ

    where  rho_tot = ½φ̇² + ½m²φ²  + ρ_m/a³ + ρ_r/a⁴ + ρ_Λ
    """
    a, phi, phi_dot = y
    a_safe = max(a, 1e-60)
    rho_ax  = 0.5 * phi_dot**2 + 0.5 * m2 * phi**2
    rho_tot = rho_ax + rho_m0/a_safe**3 + rho_r0/a_safe**4 + rho_l0
    H = (1.0 / _SQRT3) * np.sqrt(max(rho_tot, 0.0))
    a_dot   = a * H
    phi_ddot = -_SQRT3 * np.sqrt(max(rho_tot, 0.0)) * phi_dot - m2 * phi
    return [a_dot, phi_dot, phi_ddot]


def solve_ode(m_sim, phi0, phi_dot0=0.0, a0=A_IN, t_end_sim=T_FI,
              rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
              n_points=10000) -> dict:
    """High-accuracy ODE solve. Returns dict with t, a, phi, phi_dot, H, rho_ax."""
    m2 = m_sim**2
    t0_safe = max(a0, 1e-13)
    t_eval  = np.logspace(np.log10(t0_safe), np.log10(t_end_sim), n_points)
    t_start = time.time()
    sol = solve_ivp(
        lambda t, y: _one_axion_rhs(t, y, m2, rho_m0, rho_r0, rho_l0),
        [t_eval[0], t_eval[-1]], [a0, phi0, phi_dot0],
        t_eval=t_eval, method='DOP853', rtol=1e-11, atol=1e-13)
    elapsed = time.time() - t_start
    print(f"ODE: {elapsed:.2f}s  success={sol.success}  nfev={sol.nfev}")
    a, phi, phi_dot = sol.y
    rho_ax  = 0.5 * phi_dot**2 + 0.5 * m2 * phi**2
    rho_tot = rho_ax + rho_m0/a**3 + rho_r0/a**4 + rho_l0
    H = np.sqrt(rho_tot / 3.0)
    return dict(t=sol.t, a=a, phi=phi, phi_dot=phi_dot,
                rho_ax=rho_ax, H=H,
                rho_m=rho_m0/a**3, rho_r=rho_r0/a**4, rho_l=rho_l0)

## 4. Network Architecture

### แก้ไขหลัก: Log-time Input

**ปัญหา V1**: `t ∈ [1e-8, 1]` → Fourier features `sin(s·t)` ครอบ scale นี้ไม่ได้ เพราะต้องการทั้ง  
- high frequency ในช่วงต้น (t ≈ 1e-8)
- low frequency ในช่วงท้าย (t ≈ 1)

**แก้ด้วย log-time**: `τ = log(1 + t/t_scale)` ซึ่ง monotonically maps `[0, ∞) → [0, ∞)` และ expand ช่วง early time

### ScaleFactorNet ที่ถูกต้อง

IC: `a(0) = a0`, and `ȧ(0) = a0 · H(0)` where `H(0) = (1/√3)√ρ_tot(0)`

$$a(\tau) = a_0\,\exp\!\Big(\int_0^\tau H(\tau')\,d\tau'\Big) \approx a_0 \cdot \exp\!\big(\mathrm{net}(\tau) \cdot \tau\big)$$

ซึ่ง enforce `a(0) = a0` โดยอัตโนมัติ และ `a(t) > 0` เสมอ

In [ ]:
# ── Building blocks ───────────────────────────────────────────────────────────

class LogTimeFourierEmb(nn.Module):
    """
    Multi-scale Fourier embedding applied to LOG-TIME τ = log(1 + t / t_scale).

    This is the key fix: τ ∈ [0, ~18] is a slowly-varying 'stretched' time
    that allows Fourier features to cover the vast dynamic range of t ∈ [a0, 1].
    """
    def __init__(self, n_per_band: int, n_bands: int, m_sim: float,
                 t_scale: float = T_SCALE,
                 dtype=torch.float64):
        super().__init__()
        self.t_scale = float(t_scale)
        n_bands = max(2, n_bands)
        tau_max = np.log1p(1.0 / t_scale)   # log(1 + T_FI/t_scale) ≈ 18

        # Log-spaced initial scales covering [1/tau_max, m_sim]
        log_lo = np.log(1.0 / tau_max)
        log_hi = np.log(max(m_sim, 1.0))
        log_inits = np.linspace(log_lo, log_hi, n_bands)

        self.log_scales = nn.Parameter(
            torch.tensor(log_inits, dtype=dtype))            # (n_bands,)
        B = torch.randn(n_bands, n_per_band, 1, dtype=dtype)
        self.register_buffer('B', B)                         # (n_bands, n_per_band, 1)
        self.n_bands    = n_bands
        self.n_per_band = n_per_band
        self.out_dim    = n_bands * 2 * n_per_band

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # Map to log-time
        tau = torch.log1p(t / self.t_scale)      # (N, 1)
        scales = torch.exp(self.log_scales)       # (n_bands,)
        parts = []
        for b in range(self.n_bands):
            proj = 2.0 * np.pi * scales[b] * (tau @ self.B[b].T)  # (N, n_per_band)
            parts.append(torch.sin(proj))
            parts.append(torch.cos(proj))
        return torch.cat(parts, dim=-1)           # (N, out_dim)


class SinResBlock(nn.Module):
    """SIREN-style residual block."""
    def __init__(self, width: int, dtype=torch.float64):
        super().__init__()
        self.fc1 = nn.Linear(width, width, dtype=dtype)
        self.fc2 = nn.Linear(width, width, dtype=dtype)
        nn.init.uniform_(self.fc1.weight, -np.sqrt(6/width), np.sqrt(6/width))
        nn.init.uniform_(self.fc2.weight, -np.sqrt(6/width), np.sqrt(6/width))

    def forward(self, x):
        return x + self.fc2(torch.sin(self.fc1(x)))


class ScaleFactorNet(nn.Module):
    """
    Network for a(t) with hard IC:  a(t) = a0 * exp(τ · softplus(net(τ)))

    where τ = log(1 + t/t_scale) is the log-time input.

    Guarantees:
      - a(0) = a0  (since exp(0) = 1)
      - a(t) > 0   always (since exp > 0)
      - ȧ(0) = a0 · softplus(net(0)) > 0  (always expanding)

    This is a critical fix from V1 where the init gave ȧ(0) ≈ 0.693
    instead of the physical ~a0·H0 ≈ 3e5.
    """
    def __init__(self, a0: float, n_per_band: int = 16, n_bands: int = 3,
                 hidden: int = 64, depth: int = 3,
                 t_scale: float = T_SCALE, dtype=torch.float64):
        super().__init__()
        self.emb = LogTimeFourierEmb(n_per_band, n_bands, m_sim=5.0,
                                      t_scale=t_scale, dtype=dtype)
        D = self.emb.out_dim
        layers = [nn.Linear(D, hidden, dtype=dtype), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden, dtype=dtype), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1, dtype=dtype))
        self.net = nn.Sequential(*layers)
        self.sp  = nn.Softplus()
        self.register_buffer('a0', torch.tensor([[a0]], dtype=dtype))
        self.t_scale = float(t_scale)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        tau = torch.log1p(t / self.t_scale)   # log-time
        return self.a0 * torch.exp(tau * self.sp(self.net(self.emb(t))))


class WKBFieldNet(nn.Module):
    """
    WKB decomposition  φ(t) = A(t) · cos(m_sim · t + δΦ(t))

    where the slow envelopes satisfy:
        A(t)    = phi0 + phi_dot0·t + t²·A_raw(τ)
        δΦ(t)   = t · δΦ_raw(τ)

    with τ = log(1 + t/t_scale) as network input.

    Hard ICs: φ(0) = phi0 ✓,  φ̇(0) = phi_dot0 ✓
    """
    def __init__(self, phi0: float, phi_dot0: float = 0.0,
                 m_sim: float = 100.0,
                 n_per_band: int = 20, n_bands: int = 4,
                 hidden: int = 128, depth: int = 3,
                 t_scale: float = T_SCALE, dtype=torch.float64):
        super().__init__()
        self.m_sim   = float(m_sim)
        self.t_scale = float(t_scale)
        self.register_buffer('phi0',     torch.tensor([[float(phi0)]],     dtype=dtype))
        self.register_buffer('phi_dot0', torch.tensor([[float(phi_dot0)]], dtype=dtype))

        # Use slow embedding for A and δΦ (they vary on timescale ~1/H, not ~1/m)
        m_slow = max(1.0, m_sim**0.4)
        self.emb = LogTimeFourierEmb(n_per_band, n_bands, m_slow, t_scale, dtype=dtype)
        D = self.emb.out_dim

        trunk = [nn.Linear(D, hidden, dtype=dtype)]
        for _ in range(depth):
            trunk.append(SinResBlock(hidden, dtype=dtype))
        self.trunk = nn.Sequential(*trunk)
        self.head_A    = nn.Linear(hidden, 1, dtype=dtype)
        self.head_dPhi = nn.Linear(hidden, 1, dtype=dtype)

    def get_envelope_phase(self, t: torch.Tensor):
        h = self.trunk(self.emb(t))
        A_raw    = self.head_A(h)
        dPhi_raw = self.head_dPhi(h)
        A    = self.phi0 + self.phi_dot0 * t + t**2 * A_raw
        dPhi = t * dPhi_raw
        return A, dPhi

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        A, dPhi = self.get_envelope_phase(t)
        return A * torch.cos(self.m_sim * t + dPhi)


class FullICFieldNet(nn.Module):
    """
    Standard HARD IC for both φ(0) and φ̇(0):
        φ(t) = phi0 + phi_dot0·t + t²·net(τ)
    Uses LogTimeFourierEmb.
    """
    def __init__(self, phi0: float, phi_dot0: float = 0.0,
                 m_sim: float = 100.0,
                 n_per_band: int = 20, n_bands: int = 4,
                 hidden: int = 128, depth: int = 4,
                 t_scale: float = T_SCALE, dtype=torch.float64):
        super().__init__()
        self.register_buffer('phi0',     torch.tensor([[float(phi0)]],     dtype=dtype))
        self.register_buffer('phi_dot0', torch.tensor([[float(phi_dot0)]], dtype=dtype))
        self.emb  = LogTimeFourierEmb(n_per_band, n_bands, m_sim, t_scale, dtype=dtype)
        D = self.emb.out_dim
        layers = [nn.Linear(D, hidden, dtype=dtype)]
        for _ in range(depth):
            layers.append(SinResBlock(hidden, dtype=dtype))
        self.net  = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1, dtype=dtype)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.phi0 + self.phi_dot0 * t + t**2 * self.head(self.net(self.emb(t)))


class GeneralPINN(nn.Module):
    def __init__(self, a_net: ScaleFactorNet, phi_net):
        super().__init__()
        self.a_net   = a_net
        self.phi_net = phi_net

    def forward(self, t: torch.Tensor):
        return self.a_net(t), self.phi_net(t)


# ── Factory functions ─────────────────────────────────────────────────────────

def build_wkb_pinn(mc: dict, phi0: float, phi_dot0: float = 0.0,
                   a0: float = A_IN,
                   n_per_band_a: int = 16, n_per_band_phi: int = 20,
                   n_bands_a: int = 3,    n_bands_phi: int = 4,
                   hidden_a: int = 64,    hidden_phi: int = 128,
                   depth_a: int = 3,      depth_phi: int = 3,
                   dtype=torch.float64) -> GeneralPINN:
    a_net   = ScaleFactorNet(a0, n_per_band_a, n_bands_a, hidden_a, depth_a, dtype=dtype)
    phi_net = WKBFieldNet(phi0, phi_dot0, mc['m_sim'],
                          n_per_band_phi, n_bands_phi, hidden_phi, depth_phi, dtype=dtype)
    return GeneralPINN(a_net, phi_net)


def build_window_pinn(a0: float, phi0: float, phi_dot0: float,
                      m_sim: float, dt: float,
                      n_per_band_a: int = 16, n_per_band_phi: int = 20,
                      n_bands_a: int = 3,    n_bands_phi: int = 4,
                      hidden_a: int = 64,    hidden_phi: int = 128,
                      depth_a: int = 3,      depth_phi: int = 4,
                      dtype=torch.float64) -> GeneralPINN:
    # For windows, t_scale = dt/200 to stretch the local time coordinate
    t_scale = max(dt / 200.0, 1e-14)
    a_net   = ScaleFactorNet(a0, n_per_band_a, n_bands_a, hidden_a, depth_a,
                              t_scale=t_scale, dtype=dtype)
    phi_net = FullICFieldNet(phi0, phi_dot0, m_sim,
                              n_per_band_phi, n_bands_phi, hidden_phi, depth_phi,
                              t_scale=t_scale, dtype=dtype)
    return GeneralPINN(a_net, phi_net)


# ── Sanity check ──────────────────────────────────────────────────────────────
for (_phi0, _pdot0, _m) in [(1.0, 0.0, 200.0), (-0.5, 0.3, 100.0)]:
    _wkb = WKBFieldNet(_phi0, _pdot0, _m)
    _fic = FullICFieldNet(_phi0, _pdot0, _m)
    _t0  = torch.zeros(1, 1, dtype=torch.float64, requires_grad=True)
    _pw  = _wkb(_t0);  _pdw = torch.autograd.grad(_pw.sum(), _t0)[0]
    _pf  = _fic(_t0);  _pdf = torch.autograd.grad(_pf.sum(), _t0)[0]
    print(f"WKB:  φ(0)={_pw.item():.4f}(want {_phi0})  φ'(0)={_pdw.item():.4f}(want {_pdot0})")
    print(f"FullIC: φ(0)={_pf.item():.4f}(want {_phi0})  φ'(0)={_pdf.item():.4f}(want {_pdot0})")

# Check ScaleFactorNet a(0) = a0
_a_net = ScaleFactorNet(1e-8)
_t0    = torch.zeros(1, 1, dtype=torch.float64, requires_grad=True)
_a_val = _a_net(_t0)
print(f"\nScaleFactorNet a(0) = {_a_val.item():.3e}  (want 1e-8)")

## 5. Physics Loss (faithful to Code_CP equations)

In [ ]:
def physics_residuals(model: GeneralPINN, t: torch.Tensor,
                      m_sim: float,
                      rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0):
    """
    Compute residuals for the two governing equations from eoms.py:

    Friedmann:
        ȧ = (1/√3) · a · √(ρ_ax + ρ_m/a³ + ρ_r/a⁴ + ρ_Λ)
        Residual: R_F = ȧ/a - (1/√3)·√(ρ_tot)

    Klein-Gordon (from eoms.py deriv_wfromphi):
        φ̈ = -√3·√(ρ_tot)·φ̇ - m²·φ
        Residual: R_KG = φ̈ + √3·√(ρ_tot)·φ̇ + m²·φ

    Both normalised by their natural scale for balanced training.
    """
    t = t.requires_grad_(True)
    a, phi = model(t)

    a_t    = torch.autograd.grad(a,    t, torch.ones_like(a),    create_graph=True)[0]
    phi_t  = torch.autograd.grad(phi,  t, torch.ones_like(phi),  create_graph=True)[0]
    phi_tt = torch.autograd.grad(phi_t, t, torch.ones_like(phi_t), create_graph=True)[0]

    m2 = float(m_sim)**2

    # Clamped a for numerical stability
    a_c = torch.clamp(a, min=EPS)
    inv_a3 = 1.0 / (a_c**3)
    inv_a4 = 1.0 / (a_c**4)

    # ρ_ax = ½φ̇² + ½m²φ²
    rho_ax   = 0.5 * phi_t**2 + 0.5 * m2 * phi**2
    rho_tot  = rho_ax + rho_m0 * inv_a3 + rho_r0 * inv_a4 + rho_l0
    sqrt_rho = torch.sqrt(torch.clamp(rho_tot, min=EPS))

    # Hubble H = (1/√3) √ρ_tot
    H_val = _INV_SQRT3 * sqrt_rho

    # Friedmann residual: ȧ/a - H = 0
    R_F = a_t / a_c - H_val

    # Klein-Gordon residual: φ̈ + √3·√ρ_tot·φ̇ + m²φ = 0
    R_KG = phi_tt + _SQRT3 * sqrt_rho * phi_t + m2 * phi

    # Normalisation scales (detached → no gradient through them)
    scale_F  = (torch.abs(H_val).detach() + EPS)
    scale_KG = (m2 * torch.abs(phi).detach() + torch.abs(phi_tt).detach() + EPS)

    return R_F / scale_F, R_KG / scale_KG, H_val


# ── Strong causal weight (Wang et al. 2022) ───────────────────────────────────

def strong_causal_weights(t_sorted: torch.Tensor,
                           residuals: torch.Tensor,
                           eps: float = 10.0) -> torch.Tensor:
    """w_i = exp(-eps · Σ_{j<i} dt_j · r_j²)."""
    dt       = torch.diff(t_sorted, dim=0)             # (N-1, 1)
    r2       = (residuals[:-1]**2).detach()            # (N-1, 1)
    cum_loss = torch.cumsum(dt * r2, dim=0)            # (N-1, 1)
    cum_loss = torch.cat([torch.zeros(1, 1, dtype=t_sorted.dtype,
                                       device=t_sorted.device), cum_loss], dim=0)
    return torch.exp(-eps * cum_loss)


def total_loss_causal(model, t_sorted, m_sim,
                      rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                      w_F=1.0, w_KG=1.0, causal_eps=10.0):
    R_F, R_KG, H = physics_residuals(model, t_sorted, m_sim, rho_m0, rho_r0, rho_l0)
    r_max = torch.maximum(torch.abs(R_F.detach()), torch.abs(R_KG.detach()))
    cw    = strong_causal_weights(t_sorted, r_max, eps=causal_eps)
    lF    = w_F  * torch.mean(cw * R_F**2)
    lKG   = w_KG * torch.mean(cw * R_KG**2)
    return lF + lKG, lF.item(), lKG.item()


# ── Adaptive weights ─────────────────────────────────────────────────────────

class AdaptiveWeights:
    def __init__(self, w_F=1.0, w_KG=1.0, update_every=200, alpha=0.9):
        self.w_F = w_F; self.w_KG = w_KG
        self.alpha = alpha; self.update_every = update_every; self._n = 0

    def step(self, lF, lKG):
        self._n += 1
        if self._n % self.update_every == 0:
            mean = (lF + lKG) / 2.0 + EPS
            self.w_F  = self.alpha*self.w_F  + (1-self.alpha)*mean/(lF  + EPS)
            self.w_KG = self.alpha*self.w_KG + (1-self.alpha)*mean/(lKG + EPS)

    def get(self):
        return self.w_F, self.w_KG


print("Physics loss functions loaded.")
print("  ✓ physics_residuals    — Friedmann + Klein-Gordon from eoms.py")
print("  ✓ strong_causal_weights — Wang et al. 2022")
print("  ✓ total_loss_causal     — combined loss")

## 6. WKB Pretraining & RAR

In [ ]:
def _sample_collocation(n: int, t_min: float, t_max: float,
                         a_osc: float, dtype, device) -> torch.Tensor:
    """Adaptive sampling: 50% log, 30% cluster near a_osc, 20% uniform."""
    n_log = int(n * 0.50); n_cl = int(n * 0.30); n_uni = n - n_log - n_cl
    t_safe = max(t_min, 1e-14)
    t_log = np.logspace(np.log10(t_safe), np.log10(t_max), n_log)
    t_center = np.clip(a_osc, t_safe, t_max * 0.9)
    t_cl  = np.clip(np.abs(np.random.normal(t_center, t_center*0.4, n_cl)), t_safe, t_max)
    t_uni = np.random.uniform(t_safe, t_max, n_uni)
    t_all = np.sort(np.concatenate([t_log, t_cl, t_uni]))
    return torch.tensor(t_all.reshape(-1, 1), dtype=dtype, device=device)


def wkb_pretrain(model: GeneralPINN, ode_sol: dict, phi0: float,
                  n_epochs: int = 500, lr: float = 1e-3,
                  device=DEVICE, dtype=DTYPE) -> list:
    """
    Pre-train WKBFieldNet.A(t) to match analytic WKB envelope:
        A_WKB(t) = phi0 · (a0/a(t))^{3/2}

    This is the PHYSICALLY MOTIVATED initialization — the axion amplitude
    decays as a^{-3/2} in matter-dominated era (WKB approximation).
    """
    if not isinstance(model.phi_net, WKBFieldNet):
        print("  WKB pretraining skipped (not WKBFieldNet).")
        return []

    model.train()
    t_ref = torch.tensor(ode_sol['t'].reshape(-1, 1), dtype=dtype, device=device)
    a_ref = torch.tensor(ode_sol['a'].reshape(-1, 1), dtype=dtype, device=device)
    a0_val = float(ode_sol['a'][0])

    # Analytic WKB target: A ∝ a^{-3/2}
    A_target = phi0 * (a0_val / (a_ref + EPS))**1.5   # (N, 1)

    opt = optim.Adam(model.phi_net.parameters(), lr=lr, eps=1e-9)
    losses = []
    for epoch in range(n_epochs):
        opt.zero_grad()
        A_pred, _ = model.phi_net.get_envelope_phase(t_ref)
        loss = torch.mean((A_pred - A_target.detach())**2)
        loss.backward()
        nn.utils.clip_grad_norm_(model.phi_net.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

    print(f"  WKB pretrain: {n_epochs} epochs → final loss = {losses[-1]:.3e}")
    return losses


def rar_add_points(model: GeneralPINN, current_t: torch.Tensor,
                   m_sim: float, t_end_sim: float,
                   n_add: int = 200,
                   rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                   device=DEVICE, dtype=DTYPE) -> torch.Tensor:
    """RAR: add points proportional to residual magnitude."""
    model.eval()
    t_safe = max(current_t.min().item(), 1e-14)
    t_dense = torch.logspace(np.log10(t_safe), np.log10(t_end_sim), 2000,
                              dtype=dtype, device=device).reshape(-1, 1)
    with torch.no_grad():
        t_d  = t_dense.clone().requires_grad_(True)
        R_F, R_KG, _ = physics_residuals(model, t_d, m_sim, rho_m0, rho_r0, rho_l0)
        residuals = (R_F**2 + R_KG**2).detach().flatten()

    probs = (residuals / (residuals.sum() + EPS)).cpu().numpy()
    idx   = np.random.choice(len(probs), size=n_add, replace=True, p=probs)
    t_new = t_dense[idx].detach().clone()
    t_upd = torch.cat([current_t, t_new], dim=0)
    model.train()
    return t_upd[torch.argsort(t_upd[:, 0])]


print("Pretraining & RAR tools loaded.")

## 7. PCGrad (Gradient Surgery)

In [ ]:
def pcgrad_step(model: GeneralPINN, t_sorted: torch.Tensor,
                m_sim: float,
                rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                w_F=1.0, w_KG=1.0, causal_eps=10.0,
                optimizer=None):
    """
    PCGrad: project conflicting gradients.
    Returns (total_loss, lF, lKG, conflict_ratio).
    """
    params = list(model.parameters())

    # Compute causal weights once (detached)
    with torch.no_grad():
        t_c = t_sorted.clone().requires_grad_(True)
        R_F0, R_KG0, _ = physics_residuals(model, t_c, m_sim, rho_m0, rho_r0, rho_l0)
        r_max = torch.maximum(torch.abs(R_F0), torch.abs(R_KG0))
    cw = strong_causal_weights(t_sorted, r_max.detach(), eps=causal_eps)

    # Gradient for Friedmann loss
    optimizer.zero_grad()
    R_F1, _, _ = physics_residuals(model, t_sorted, m_sim, rho_m0, rho_r0, rho_l0)
    lF_t = w_F * torch.mean(cw * R_F1**2)
    lF_t.backward(retain_graph=True)
    g_F  = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]

    # Gradient for Klein-Gordon loss
    optimizer.zero_grad()
    _, R_KG2, _ = physics_residuals(model, t_sorted, m_sim, rho_m0, rho_r0, rho_l0)
    lKG_t = w_KG * torch.mean(cw * R_KG2**2)
    lKG_t.backward()
    g_KG = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]

    # PCGrad projection
    n_conflict = 0
    g_F_pc, g_KG_pc = [], []
    for gf, gkg in zip(g_F, g_KG):
        gf_f  = gf.view(-1)
        gkg_f = gkg.view(-1)
        dot   = (gf_f * gkg_f).sum()
        if dot < 0:
            n_conflict += 1
            gf_f  = gf_f  - dot / (gkg_f.norm()**2 + EPS) * gkg_f
            gkg_f = gkg_f - dot / (gf.view(-1).norm()**2 + EPS) * gf.view(-1)
        g_F_pc.append(gf_f.view_as(gf))
        g_KG_pc.append(gkg_f.view_as(gkg))

    optimizer.zero_grad()
    for p, gf, gkg in zip(params, g_F_pc, g_KG_pc):
        p.grad = gf + gkg

    nn.utils.clip_grad_norm_(params, max_norm=1.0)
    optimizer.step()

    return (lF_t.item() + lKG_t.item(), lF_t.item(), lKG_t.item(),
            n_conflict / max(len(params), 1))


print("PCGrad loaded.")

## 8. Training Pipeline

In [ ]:
def train_advanced(
    model: GeneralPINN,
    mc: dict,
    ode_sol: dict,
    phi0: float,
    # Training sizes
    n_colloc: int    = 4000,
    batch_size: int  = 512,
    # Adam
    adam_epochs: int = 10000,
    adam_lr: float   = 3e-4,
    # L-BFGS
    lbfgs_iters: int = 3000,
    # WKB pretrain
    pretrain_epochs: int = 500,
    pretrain_lr: float   = 1e-3,
    # Causal
    causal_eps: float = 10.0,
    # Flags
    use_pcgrad: bool  = True,
    use_rar: bool     = True,
    rar_every: int    = 1000,
    rar_add: int      = 200,
    rar_max: int      = 8000,
    print_every: int  = 1000,
    rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Full training pipeline:
      0. WKB Pretraining (analytic envelope init)
      1. Adam + Strong Causal + PCGrad + RAR
      2. L-BFGS refinement
    """
    m_sim     = mc['m_sim']
    t_end_sim = mc['t_end_sim']
    a_osc     = mc['a_osc']

    hist = dict(adam_loss=[], lF=[], lKG=[], w_F=[], w_KG=[],
                conflict_ratio=[], n_colloc_history=[],
                pretrain_loss=[], lbfgs_loss=[])

    # ── Phase 0: WKB Pretraining ──────────────────────────────────────────────
    if pretrain_epochs > 0:
        print("\n── Phase 0: WKB Pretraining ─────────────────────────────────")
        pre = wkb_pretrain(model, ode_sol, phi0, pretrain_epochs, pretrain_lr, device, dtype)
        hist['pretrain_loss'] = pre

    # ── Phase 1: Adam ────────────────────────────────────────────────────────
    aw  = AdaptiveWeights(update_every=200)
    opt = optim.Adam(model.parameters(), lr=adam_lr, eps=1e-9, betas=(0.9, 0.999))
    sch = CosineAnnealingLR(opt, T_max=adam_epochs, eta_min=adam_lr * 1e-3)

    print(f"\n── Phase 1: Adam ({adam_epochs} epochs)  "
          f"PCGrad={use_pcgrad}  RAR={use_rar}  causal_eps={causal_eps} ──")

    t_pool = _sample_collocation(n_colloc, 0.0, t_end_sim, a_osc, dtype, device)
    t_pool = t_pool[torch.argsort(t_pool[:, 0])]

    t0 = time.time()
    for epoch in range(adam_epochs):
        idx   = torch.sort(torch.randperm(len(t_pool), device=device)[:batch_size])[0]
        t_b   = t_pool[idx].clone()
        w_F, w_KG = aw.get()

        if use_pcgrad:
            loss_v, lF, lKG, cr = pcgrad_step(
                model, t_b, m_sim, rho_m0, rho_r0, rho_l0,
                w_F=w_F, w_KG=w_KG, causal_eps=causal_eps, optimizer=opt)
        else:
            opt.zero_grad()
            loss, lF, lKG = total_loss_causal(
                model, t_b, m_sim, rho_m0, rho_r0, rho_l0,
                w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_v = loss.item()
            cr = 0.0

        sch.step(); aw.step(lF, lKG)

        hist['adam_loss'].append(loss_v)
        hist['lF'].append(lF); hist['lKG'].append(lKG)
        hist['w_F'].append(w_F); hist['w_KG'].append(w_KG)
        hist['conflict_ratio'].append(cr)
        hist['n_colloc_history'].append(len(t_pool))

        if use_rar and epoch > 0 and epoch % rar_every == 0 and len(t_pool) < rar_max:
            t_pool = rar_add_points(model, t_pool, m_sim, t_end_sim,
                                     n_add=rar_add, rho_m0=rho_m0, rho_r0=rho_r0,
                                     rho_l0=rho_l0, device=device, dtype=dtype)

        if epoch % print_every == 0 or epoch == adam_epochs - 1:
            avg_cr = float(np.mean(hist['conflict_ratio'][-100:])) if hist['conflict_ratio'] else 0
            print(f"  [{epoch:5d}] loss={loss_v:.3e}  lF={lF:.3e}  "
                  f"lKG={lKG:.3e}  conflict={avg_cr:.2f}  "
                  f"n_col={len(t_pool)}  lr={opt.param_groups[0]['lr']:.2e}")

    print(f"  Adam done {time.time()-t0:.1f}s")

    # ── Phase 2: L-BFGS ───────────────────────────────────────────────────────
    print(f"\n── Phase 2: L-BFGS ({lbfgs_iters} iters) ────────────────────────")
    t_full = t_pool.clone().requires_grad_(True)
    w_F, w_KG = aw.get()
    lb_buf: list = []

    opt_lb = optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters,
                          tolerance_grad=1e-11, tolerance_change=1e-12,
                          history_size=200, line_search_fn='strong_wolfe')

    def _closure():
        opt_lb.zero_grad()
        loss, lF_, lKG_ = total_loss_causal(
            model, t_full, m_sim, rho_m0, rho_r0, rho_l0,
            w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
        loss.backward()
        lb_buf.append(loss.item())
        return loss

    t0 = time.time()
    opt_lb.step(_closure)
    hist['lbfgs_loss'] = lb_buf
    print(f"  L-BFGS done {time.time()-t0:.1f}s  final={lb_buf[-1]:.3e}")

    return hist


print("train_advanced() loaded.")

## 9. Sequential Window Training

In [ ]:
def _extract_ic(model: GeneralPINN, dt: float, device=DEVICE, dtype=DTYPE):
    """Extract (a, φ, φ̇) at local time = dt via autograd."""
    t_end_t = torch.tensor([[dt]], dtype=dtype, device=device, requires_grad=True)
    a_end, phi_end = model(t_end_t)
    phi_dot_end = torch.autograd.grad(
        phi_end, t_end_t, torch.ones_like(phi_end))[0]
    return (a_end.detach().item(),
            phi_end.detach().item(),
            phi_dot_end.detach().item())


def sequential_train(
    mc: dict, phi0: float, a0: float = A_IN, phi_dot0: float = 0.0,
    n_windows: int = 5,
    n_per_band_phi: int = 20, n_bands_phi: int = 4,
    n_per_band_a:   int = 16, n_bands_a:   int = 3,
    hidden_phi: int = 128, hidden_a: int = 64,
    depth_phi: int = 4,   depth_a: int = 3,
    n_colloc: int = 3000, batch_size: int = 512,
    adam_epochs: int = 5000, adam_lr: float = 5e-4,
    lbfgs_iters: int = 1000,
    causal_eps: float = 5.0,
    use_pcgrad: bool = True, use_rar: bool = True,
    print_every: int = 5000,
    rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
    device=DEVICE, dtype=DTYPE,
) -> list:
    """
    Train K sequential windows across [0, t_end_sim].
    Each window uses FullICFieldNet with hard IC for both φ and φ̇.
    IC for window k+1 is extracted from window k via autograd.
    """
    t_end_sim = mc['t_end_sim']
    m_sim     = mc['m_sim']
    t_edges   = np.linspace(0.0, t_end_sim, n_windows + 1)

    results = []
    cur_a0, cur_phi0, cur_phidot0 = a0, phi0, phi_dot0

    print(f"\n{'='*60}")
    print(f"  Sequential: {n_windows} windows  m_sim={m_sim:.3g}  T={t_end_sim:.3e}")
    print(f"  N_osc/window ≈ {m_sim/(2*np.pi)*t_end_sim/n_windows:.1f}")
    print(f"{'='*60}")

    for k in range(n_windows):
        t_lo, t_hi = t_edges[k], t_edges[k + 1]
        dt = t_hi - t_lo
        mc_k = {**mc, 't_end_sim': dt, 'n_osc': m_sim/(2*np.pi)*dt}

        print(f"\n── Window {k+1}/{n_windows}  t∈[{t_lo:.3e},{t_hi:.3e}]  "
              f"N_osc≈{mc_k['n_osc']:.2f}  "
              f"a={cur_a0:.3e}  φ={cur_phi0:.4f}")

        # For windowed training, ODE is needed for WKB pretrain
        # Here we skip pretrain for sequential windows (FullICFieldNet)
        model_k = build_window_pinn(
            cur_a0, cur_phi0, cur_phidot0, m_sim, dt,
            n_per_band_a=n_per_band_a, n_per_band_phi=n_per_band_phi,
            n_bands_a=n_bands_a, n_bands_phi=n_bands_phi,
            hidden_a=hidden_a, hidden_phi=hidden_phi,
            depth_a=depth_a, depth_phi=depth_phi, dtype=dtype,
        ).to(device, dtype=dtype)

        # Create dummy ode_sol to skip pretrain in train_advanced
        hist_k = train_advanced(
            model_k, mc_k, None, cur_phi0,
            n_colloc=n_colloc, batch_size=batch_size,
            adam_epochs=adam_epochs, adam_lr=adam_lr,
            lbfgs_iters=lbfgs_iters,
            pretrain_epochs=0,    # No WKB pretrain for FullIC net
            causal_eps=causal_eps,
            use_pcgrad=use_pcgrad, use_rar=use_rar,
            print_every=print_every,
            rho_m0=rho_m0, rho_r0=rho_r0, rho_l0=rho_l0,
            device=device, dtype=dtype,
        )

        cur_a0, cur_phi0, cur_phidot0 = _extract_ic(model_k, dt, device, dtype)
        results.append(dict(model=model_k, t_lo=t_lo, t_hi=t_hi,
                            history=hist_k, dt=dt))

    print(f"\n  Sequential training complete. {n_windows} windows.")
    return results


def sequential_predict(window_results: list, t_eval: np.ndarray,
                        device=DEVICE, dtype=DTYPE):
    a_pred   = np.full_like(t_eval, np.nan)
    phi_pred = np.full_like(t_eval, np.nan)
    for wr in window_results:
        t_lo, t_hi = wr['t_lo'], wr['t_hi']
        is_last = (wr is window_results[-1])
        mask = (t_eval >= t_lo) & ((t_eval <= t_hi) if is_last else (t_eval < t_hi))
        if mask.sum() == 0:
            continue
        t_local = t_eval[mask] - t_lo
        t_t = torch.tensor(t_local.reshape(-1, 1), dtype=dtype, device=device)
        with torch.no_grad():
            a_w, phi_w = wr['model'](t_t)
        a_pred[mask]   = a_w.cpu().numpy().flatten()
        phi_pred[mask] = phi_w.cpu().numpy().flatten()
    return a_pred, phi_pred


print("Sequential training loaded.")

## 10. Evaluation & Plotting

In [ ]:
def _l2rel(pred, ref):
    return np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + EPS)


def evaluate_pinn(model_or_windows, ode_sol: dict, mc: dict,
                   n_eval: int = 5000,
                   is_sequential: bool = False,
                   device=DEVICE, dtype=DTYPE) -> dict:
    """Evaluate PINN (single or sequential) against ODE reference."""
    t_safe = max(1e-13, mc['t_end_sim'] * 1e-5)
    t_sim  = np.logspace(np.log10(t_safe), np.log10(mc['t_end_sim']), n_eval)

    if is_sequential:
        a_pred, phi_pred = sequential_predict(model_or_windows, t_sim, device, dtype)
        A_arr = dPhi_arr = None
    else:
        model = model_or_windows
        model.eval()
        t_t = torch.tensor(t_sim.reshape(-1, 1), dtype=dtype, device=device)
        with torch.no_grad():
            a_pred   = model.a_net(t_t).cpu().numpy().flatten()
            phi_pred = model.phi_net(t_t).cpu().numpy().flatten()
        A_arr = dPhi_arr = None
        if isinstance(model.phi_net, WKBFieldNet):
            with torch.no_grad():
                A_t, dPhi_t = model.phi_net.get_envelope_phase(t_t)
                A_arr    = A_t.cpu().numpy().flatten()
                dPhi_arr = dPhi_t.cpu().numpy().flatten()

    a_ode   = interp1d(ode_sol['t'], ode_sol['a'],   kind='cubic',
                        fill_value='extrapolate')(t_sim)
    phi_ode = interp1d(ode_sol['t'], ode_sol['phi'], kind='cubic',
                        fill_value='extrapolate')(t_sim)

    metrics = dict(
        L2_a    = _l2rel(a_pred, a_ode),
        L2_phi  = _l2rel(phi_pred, phi_ode),
        max_a   = np.max(np.abs(a_pred - a_ode)),
        max_phi = np.max(np.abs(phi_pred - phi_ode)),
    )
    print(f"  L2(a)={metrics['L2_a']:.3e}  L2(φ)={metrics['L2_phi']:.3e}  "
          f"max(a)={metrics['max_a']:.3e}  max(φ)={metrics['max_phi']:.3e}")

    t_phys = t_sim * mc['lam']
    return dict(t_sim=t_sim, t_phys=t_phys,
                a_pred=a_pred, phi_pred=phi_pred,
                a_ode=a_ode,   phi_ode=phi_ode,
                A=A_arr, dPhi=dPhi_arr, metrics=metrics)


def plot_diagnostics(ev: dict, mc: dict, history: dict = None,
                      title: str = '', save_dir: str = 'results_v2'):
    os.makedirs(save_dir, exist_ok=True)

    t_phys  = ev['t_phys']
    a_pred  = ev['a_pred']
    phi_pred= ev['phi_pred']
    a_ode   = ev['a_ode']
    phi_ode = ev['phi_ode']

    n_rows = 3 if (ev.get('A') is not None or history is not None) else 2
    fig, axes = plt.subplots(n_rows, 3, figsize=(17, 5*n_rows), dpi=110)

    # Row 0: Scale factor
    axes[0,0].loglog(t_phys, a_ode, 'k-', lw=2, label='ODE')
    axes[0,0].loglog(t_phys, a_pred, 'C1--', lw=1.5, label='PINN')
    axes[0,0].set(xlabel='t', ylabel='a(t)', title='Scale Factor')
    axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

    # Row 0: a relative error
    err_a = np.abs(a_pred - a_ode) / (np.abs(a_ode) + EPS) * 100
    axes[0,1].semilogx(t_phys, err_a, 'C1', lw=1)
    axes[0,1].axhline(1.0, color='grey', ls='--', lw=1, label='1%')
    axes[0,1].set(xlabel='t', ylabel='Rel. Err (%)', title='a(t) Relative Error')
    axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

    # Row 0: phi early-time comparison
    mask_early = t_phys < t_phys[len(t_phys)//6]
    axes[0,2].plot(t_phys[mask_early], phi_ode[mask_early], 'k-', lw=1.8, label='ODE')
    axes[0,2].plot(t_phys[mask_early], phi_pred[mask_early], 'C1--', lw=1.2, label='PINN')
    axes[0,2].set(xlabel='t', ylabel='φ(t)', title='Axion Field (early time)')
    axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

    # Row 1: phi relative error, full phi, metrics
    err_phi = np.abs(phi_pred - phi_ode) / (np.abs(phi_ode) + EPS) * 100
    axes[1,0].semilogx(t_phys, err_phi + EPS, 'coral', lw=1)
    axes[1,0].axhline(1.0, color='grey', ls='--', lw=1, label='1%')
    axes[1,0].set(xlabel='t', ylabel='Rel. Err (%)', title='φ(t) Relative Error')
    axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

    axes[1,1].semilogx(t_phys, phi_ode, 'k-', lw=1.8, label='ODE')
    axes[1,1].semilogx(t_phys, phi_pred, 'C1--', lw=1.2, label='PINN', alpha=0.8)
    axes[1,1].set(xlabel='t', ylabel='φ(t)', title='Axion Field (full range)')
    axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

    m = ev['metrics']
    info = (f"m_target = {mc['m_target']:.3g}  ({mc['regime']})\n"
            f"mode     = {mc['mode']}\n\n"
            f"L2_rel_a    = {m['L2_a']:.3e}\n"
            f"MaxAbs_a    = {m['max_a']:.3e}\n"
            f"L2_rel_phi  = {m['L2_phi']:.3e}\n"
            f"MaxAbs_phi  = {m['max_phi']:.3e}")
    axes[1,2].text(0.05, 0.95, info, ha='left', va='top',
                   transform=axes[1,2].transAxes,
                   fontfamily='monospace', fontsize=9.5,
                   bbox=dict(boxstyle='round', fc='#f7f7e8', alpha=0.9))
    axes[1,2].set_axis_off(); axes[1,2].set_title('Error Metrics Summary')

    # Row 2 (optional): WKB envelope + training loss
    if n_rows == 3:
        if ev.get('A') is not None:
            axes[2,0].semilogx(t_phys, ev['A'], 'C1', lw=1.5, label='PINN A(t)')
            A_wkb = np.abs(phi_ode[0]) * (a_ode[0] / (a_ode + EPS))**1.5
            axes[2,0].semilogx(t_phys, A_wkb, 'k:', lw=1.2, label=r'WKB $\propto a^{-3/2}$')
            axes[2,0].set(xlabel='t', ylabel='A(t)', title='WKB Envelope')
            axes[2,0].legend(); axes[2,0].grid(alpha=0.3)

            axes[2,1].semilogx(t_phys, ev['dPhi'], 'C2', lw=1.5)
            axes[2,1].set(xlabel='t', ylabel=r'$\delta\Phi(t)$', title='Phase Correction')
            axes[2,1].grid(alpha=0.3)

        if history is not None and history.get('adam_loss'):
            ax_loss = axes[2, 2 if ev.get('A') is not None else 0]
            ax_loss.semilogy(history['adam_loss'], 'C0', lw=1, label='Adam')
            ax_loss.set(xlabel='epoch', ylabel='loss', title='Training Loss (Adam)')
            ax_loss.legend(); ax_loss.grid(alpha=0.3)

            if history.get('lbfgs_loss'):
                ax2 = axes[2, 1 if ev.get('A') is not None else 1]
                ax2.semilogy(history['lbfgs_loss'], 'C3', lw=1.2, label='L-BFGS')
                ax2.set(xlabel='iteration', ylabel='loss', title='L-BFGS Loss')
                ax2.legend(); ax2.grid(alpha=0.3)

    ttl = title or f'PINN V2 — m = {mc["m_target"]:.3g}  ({mc["regime"]})'
    fig.suptitle(ttl, fontsize=13, fontweight='bold')
    plt.tight_layout()

    tag  = f"m{mc['m_target']:.0e}".replace('+', '').replace('e0', 'e')
    path = os.path.join(save_dir, f'{tag}_v2.png')
    plt.savefig(path, bbox_inches='tight', dpi=120)
    plt.show()
    print(f"Saved → {path}")


print("Evaluation & plotting loaded.")

## 11. Master Pipelines

In [ ]:
def run_wkb(
    m_target: float = 200.0,
    phi0: float = 1.0, a0: float = A_IN, t_end: float = T_FI,
    # Network
    n_per_band_a: int = 16, n_per_band_phi: int = 20,
    n_bands_a: int = 3,     n_bands_phi: int = 4,
    hidden_a: int = 64,     hidden_phi: int = 128,
    depth_a: int = 3,       depth_phi: int = 3,
    # Training
    adam_epochs: int    = 10000,
    adam_lr: float      = 3e-4,
    lbfgs_iters: int    = 3000,
    pretrain_epochs:int = 500,
    n_colloc: int       = 4000,
    batch_size: int     = 512,
    causal_eps: float   = 10.0,
    use_pcgrad: bool    = True,
    use_rar: bool       = True,
    print_every: int    = 2000,
    save_dir: str       = 'results_v2',
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """Full WKB PINN pipeline with all V2 improvements."""
    mc = analyze_mass_regime(m_target, t_end)
    print_mass_summary(mc)

    print(f"\nSolving ODE reference (m_sim={mc['m_sim']:.3g}) ...")
    ode_sol = solve_ode(mc['m_sim'], phi0, 0.0, a0, mc['t_end_sim'])

    # Auto-scale network to oscillation count
    n_osc = mc['m_sim'] / (2*np.pi) * mc['t_end_sim']
    if n_osc > 50:
        n_per_band_phi = max(n_per_band_phi, 28)
        hidden_phi     = max(hidden_phi, 160)
        depth_phi      = max(depth_phi, 4)

    model = build_wkb_pinn(mc, phi0, 0.0, a0,
                            n_per_band_a, n_per_band_phi,
                            n_bands_a,   n_bands_phi,
                            hidden_a, hidden_phi, depth_a, depth_phi,
                            dtype=dtype).to(device, dtype=dtype)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nWKB PINN V2  params={n_params:,}")

    history = train_advanced(
        model, mc, ode_sol, phi0,
        n_colloc=n_colloc, batch_size=batch_size,
        adam_epochs=adam_epochs, adam_lr=adam_lr,
        lbfgs_iters=lbfgs_iters,
        pretrain_epochs=pretrain_epochs,
        causal_eps=causal_eps,
        use_pcgrad=use_pcgrad, use_rar=use_rar,
        print_every=print_every,
        device=device, dtype=dtype,
    )

    ev = evaluate_pinn(model, ode_sol, mc, device=device, dtype=dtype)
    plot_diagnostics(ev, mc, history, save_dir=save_dir)
    return dict(mc=mc, model=model, ode_sol=ode_sol, history=history, eval=ev)


def run_sequential(
    m_target: float = 200.0,
    phi0: float = 1.0, a0: float = A_IN, t_end: float = T_FI,
    n_windows: int = 5,
    n_per_band_a: int = 16, n_per_band_phi: int = 20,
    n_bands_a: int = 3,     n_bands_phi: int = 4,
    hidden_a: int = 64,     hidden_phi: int = 128,
    depth_a: int = 3,       depth_phi: int = 4,
    adam_epochs: int = 5000, adam_lr: float = 5e-4,
    lbfgs_iters: int = 1000,
    n_colloc: int = 3000, batch_size: int = 512,
    causal_eps: float = 5.0,
    use_pcgrad: bool = True, use_rar: bool = True,
    save_dir: str = 'results_v2',
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """Full Sequential PINN pipeline with V2 improvements."""
    mc = analyze_mass_regime(m_target, t_end)
    print_mass_summary(mc)

    print(f"\nSolving ODE reference ...")
    ode_sol = solve_ode(mc['m_sim'], phi0, 0.0, a0, mc['t_end_sim'])

    window_results = sequential_train(
        mc, phi0, a0, phi_dot0=0.0, n_windows=n_windows,
        n_per_band_a=n_per_band_a, n_per_band_phi=n_per_band_phi,
        n_bands_a=n_bands_a, n_bands_phi=n_bands_phi,
        hidden_a=hidden_a, hidden_phi=hidden_phi,
        depth_a=depth_a, depth_phi=depth_phi,
        n_colloc=n_colloc, batch_size=batch_size,
        adam_epochs=adam_epochs, adam_lr=adam_lr,
        lbfgs_iters=lbfgs_iters,
        causal_eps=causal_eps,
        use_pcgrad=use_pcgrad, use_rar=use_rar,
        print_every=adam_epochs,
        device=device, dtype=dtype,
    )

    ev = evaluate_pinn(window_results, ode_sol, mc,
                        is_sequential=True, device=device, dtype=dtype)
    plot_diagnostics(ev, mc, save_dir=save_dir)
    return dict(mc=mc, ode_sol=ode_sol, window_results=window_results, eval=ev)


def run_curriculum(
    m_target: float = 1e5,
    phi0: float = 1.0, a0: float = A_IN, t_end: float = T_FI,
    n_per_band_a: int = 16, n_per_band_phi: int = 20,
    n_bands_a: int = 3,     n_bands_phi: int = 4,
    hidden_a: int = 64,     hidden_phi: int = 128,
    depth_a: int = 3,       depth_phi: int = 3,
    pretrain_epochs: int = 300,
    causal_eps: float = 10.0,
    use_pcgrad: bool = True, use_rar: bool = True,
    lbfgs_final: int = 3000,
    n_colloc: int = 4000, batch_size: int = 512,
    print_every: int = 2000,
    save_dir: str = 'results_v2',
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Curriculum learning: m_target^(1/3) → m_target^(2/3) → m_target.
    Each stage inherits weights from previous.
    """
    m_stages        = [m_target**(1/3), m_target**(2/3), m_target]
    epochs_schedule = [5000, 6000, 8000]
    lr_schedule     = [5e-4, 4e-4, 3e-4]

    print(f"\n{'='*62}")
    print(f"  CURRICULUM LEARNING  m_target={m_target:.3g}")
    print(f"  Stages: {[f'{m:.2g}' for m in m_stages]}")
    print(f"{'='*62}")

    model = None
    all_hist = []

    for stage_i, (m_s, n_ep, lr_s) in enumerate(
            zip(m_stages, epochs_schedule, lr_schedule)):
        print(f"\n── Stage {stage_i+1}/{len(m_stages)}: m={m_s:.4g}  epochs={n_ep} ──")
        mc_s  = analyze_mass_regime(m_s, t_end)
        ode_s = solve_ode(mc_s['m_sim'], phi0, 0.0, a0, mc_s['t_end_sim'])

        if model is None:
            model = build_wkb_pinn(mc_s, phi0, 0.0, a0,
                                    n_per_band_a, n_per_band_phi,
                                    n_bands_a, n_bands_phi,
                                    hidden_a, hidden_phi, depth_a, depth_phi,
                                    dtype=dtype).to(device, dtype=dtype)
            n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Model built: {n_p:,} parameters")
        else:
            model.phi_net.m_sim = float(mc_s['m_sim'])
            print(f"  Warm-starting  phi_net.m_sim → {mc_s['m_sim']:.3g}")

        hist_s = train_advanced(
            model, mc_s, ode_s, phi0,
            n_colloc=n_colloc, batch_size=batch_size,
            adam_epochs=n_ep, adam_lr=lr_s, lbfgs_iters=500,
            pretrain_epochs=pretrain_epochs if stage_i == 0 else 0,
            causal_eps=causal_eps,
            use_pcgrad=use_pcgrad, use_rar=use_rar,
            print_every=print_every,
            device=device, dtype=dtype,
        )
        all_hist.append(hist_s)

    # Final ODE & L-BFGS at target
    mc_target  = analyze_mass_regime(m_target, t_end)
    ode_target = solve_ode(mc_target['m_sim'], phi0, 0.0, a0, mc_target['t_end_sim'])
    model.phi_net.m_sim = float(mc_target['m_sim'])

    print(f"\n── Final L-BFGS at m_target={m_target:.3g} ({lbfgs_final} iters) ──")
    t_full = _sample_collocation(n_colloc, 0.0, mc_target['t_end_sim'],
                                  mc_target['a_osc'], dtype, device)
    t_full = t_full[torch.argsort(t_full[:, 0])].requires_grad_(True)
    lb_buf: list = []
    opt_lb = optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_final,
                          tolerance_grad=1e-11, tolerance_change=1e-12,
                          history_size=200, line_search_fn='strong_wolfe')

    def _final_closure():
        opt_lb.zero_grad()
        loss, _, _ = total_loss_causal(model, t_full, mc_target['m_sim'],
                                        causal_eps=causal_eps)
        loss.backward(); lb_buf.append(loss.item()); return loss

    opt_lb.step(_final_closure)
    print(f"  Final L-BFGS done  final={lb_buf[-1]:.3e}")

    ev = evaluate_pinn(model, ode_target, mc_target, device=device, dtype=dtype)
    plot_diagnostics(ev, mc_target, save_dir=save_dir)
    return dict(model=model, mc=mc_target, ode_sol=ode_target,
                all_histories=all_hist, final_lbfgs_loss=lb_buf, eval=ev)


print("\n✓ Master pipelines loaded:")
print("  run_wkb()         — WKB + MultiScale + all advanced techniques")
print("  run_sequential()  — Sequential windows")
print("  run_curriculum()  — Curriculum: m^(1/3) → m^(2/3) → m")

## 12. Experiment A — m = 200 (Oscillating Regime)

N_osc ≈ 32 — direct mode. Tests that the physics equations are correct and
network architecture works. Expected: L2(a) < 0.1%, L2(φ) < 5%.

In [ ]:
print("=" * 62)
print("  EXPERIMENT A: WKB PINN V2   m = 200")
print("=" * 62)

res_A = run_wkb(
    m_target       = 200.0,
    phi0           = 1.0,
    a0             = A_IN,
    t_end          = T_FI,
    adam_epochs    = 10000,
    adam_lr        = 3e-4,
    lbfgs_iters    = 3000,
    pretrain_epochs= 500,
    n_colloc       = 4000,
    causal_eps     = 10.0,
    use_pcgrad     = True,
    use_rar        = True,
    print_every    = 2000,
)

In [ ]:
print("=" * 62)
print("  EXPERIMENT A: Sequential (5 windows)  m = 200")
print("=" * 62)

res_A_seq = run_sequential(
    m_target    = 200.0,
    phi0        = 1.0,
    n_windows   = 5,
    adam_epochs = 5000,
    adam_lr     = 5e-4,
    lbfgs_iters = 1000,
    n_colloc    = 3000,
    causal_eps  = 5.0,
)

## 13. Experiment B — m = 1×10⁵ (Rapid-Oscillation + Rescaling)

After rescaling: m_sim ≈ 251, N_osc_sim ≈ 40. Uses Curriculum Learning.

In [ ]:
print("=" * 62)
print("  EXPERIMENT B: Curriculum Learning  m = 1e5")
print("=" * 62)

res_B = run_curriculum(
    m_target       = 1e5,
    phi0           = 1.0,
    pretrain_epochs= 300,
    causal_eps     = 10.0,
    use_pcgrad     = True,
    use_rar        = True,
    lbfgs_final    = 3000,
    print_every    = 2000,
)

## 14. Summary & Comparison

In [ ]:
print("=" * 70)
print("  PINN V2 RESULTS SUMMARY")
print("=" * 70)
print(f"{'Experiment':<30}  {'Method':<20}  {'L2(a)':>10}  {'L2(φ)':>10}")
print("-" * 70)

rows = []
for label, res in [
    ("Exp A  m=200   WKB V2",       res_A),
    ("Exp A  m=200   Sequential",   res_A_seq),
    ("Exp B  m=1e5   Curriculum",   res_B),
]:
    m = res['eval']['metrics']
    La, Lp = m['L2_a'], m['L2_phi']
    flag = " ✓" if (La < 0.01 and Lp < 0.10) else (" ~" if La < 0.05 else " ✗")
    name, method = label.split('  ', 1)
    print(f"  {name:<28}  {method:<20}  {La:>10.3e}  {Lp:>10.3e}{flag}")

print("=" * 70)
print("\nKey improvements in V2:")
print("  ✓ Log-time input: τ=log(1+t/t_scale) covers full [1e-8, 1] range")
print("  ✓ ScaleFactorNet: a(t)=a0·exp(τ·net(τ)) — correct init & monotone")
print("  ✓ WKB pretraining: A(t)∝a^{-3/2} — physical starting point")
print("  ✓ Strong causal PINN (ε=10) — enforces left-to-right solving")
print("  ✓ PCGrad — removes gradient conflict between F & KG losses")
print("  ✓ RAR — focuses collocation on high-residual regions")
print("  ✓ Curriculum — m^(1/3)→m^(2/3)→m warm start for high masses")